# **Grad-CAM: Practice**

Welcome to the lesson on the GradCAM method within our [course](https://stepik.org/a/198640) on explainable artificial intelligence. In this practical part we will work with GradCAM.

GradCAM (Gradient-weighted Class Activation Mapping) is one of the most popular methods for visualising activations. Its popularity comes from the number of modifications of the method and from its simplicity (after all, it is just an add-on over class activation maps).

Just like class activation maps, the method lets you understand which parts of the image influenced the *final classification* most strongly.

In this lesson you will look at and revise how GradCAM works.

Happy coding!

<img src="https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/pawel-czerwinski-CEr4ljp-MSh4-unsplash.jpg" alt="pawel-czerwinski-CEr4ljp-MSh4-unsplash" border="0">

**Step 1. Let us import the libraries we need.**

In [ ]:
!pip install grad-cam -q

In [ ]:
import os
import cv2
import numpy as np
import requests
from io import BytesIO
from matplotlib import pyplot as plt
from matplotlib.pyplot import imshow
from PIL import Image
import torch
from torch import nn
from torchvision import models, transforms
from torch.nn import functional as F
from torch import nn as nn
from torch.autograd import Variable
from torchvision.models import resnet50, ResNet50_Weights, densenet201, DenseNet201_Weights


#For working with the GradCAM implementation under the hood
import warnings
warnings.filterwarnings('ignore')
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image, deprocess_image, preprocess_image

**A reminder of the theory.**

Let us recall what we need in order to build GradCAM.

- Get the Class Activation Map
- Weight the obtained class activation map by the gradient

Thanks to the flexibility of pyTorch, these Grad-CAM steps can be implemented in three ways:
- using hooks, which you met in the practice on Guided backpropagation;
- by redefining the last layers of the network;
- by using the method already implemented under the hood.

**What is the difference and what should you use?**

As we went through in the theoretical part, to obtain a CAM you need access to the last convolutional layer of the model. In turn, model implementations can be different, so the process of getting the feature map from the last convolutional layer has to be adapted to a particular architecture (including your own). That is why it is more convenient and faster to use [ready-made methods](https://pypi.org/project/grad-cam/).

But implementing the output values of interpretation methods by hand is useful for understanding the methods and their drawbacks "from the inside". So in this practice you will both work with the library and implement GradCAM yourself, using two architectures as an example: `ResNet` and `DenseNet`.

Let us load the models we are going to work with.

In [ ]:
densenet = densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)
resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

Call each variable one by one and look carefully at the architectures of the models.

In [ ]:
#densenet

In [ ]:
#resnet

In the previous lesson, when you were building a Class Activation Map for ResNet, you used three components of the architecture:

- The activation maps from the last convolutional layer
- Global average pooling over the activation maps
- The linear layer that predicts the probability

To build maps for *other* models, you also need access to these components of the architecture.

At the top level, every neural network can be seen as a sequence of modules that perform operations on the input object. You can get access to the modules of the network by calling the `model.children()` iterator. However, getting access to a module is not enough to reach the last convolutional layer. You have to go deeper into the model.

To do that, as was described above, you can either redefine the architecture a little, or attach a `hook` to the layer you need, or use ready-made solutions.

To make things clear, let us go through each method one by one.

First let us write the helper functions and preprocess the image so that it can be fed into the neural network.

In [ ]:
# Loading the image
url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg'

image_bytes = requests.get(url).content
image = Image.open(BytesIO(image_bytes)) # Once again we look at the particular example x_0

plt.figure(figsize=(8,10))
plt.axis('off')
plt.imshow(image);

In [ ]:
#Preprocessing

preprocess = transforms.Compose([
   transforms.Resize((224,224)),
   transforms.ToTensor(),
   transforms.Normalize(
   mean=[0.485, 0.456, 0.406],
   std=[0.229, 0.224, 0.225]
)
])

# A function for a simple resize of the image
display = transforms.Compose([
    transforms.Resize((224,224))
    ])

tensor = preprocess(image)
pred = Variable((tensor.unsqueeze(0)), requires_grad=True)

**Way 1. Redefining the model.**

Redefining the model is done by writing your own network class. When writing the new model you have to separate out two parts:

- the one that extracts the features;
- the one that performs the classification from the extracted features.  

We will store them in the class attributes `features` and `classifier` respectively.

In [ ]:
#ResNet

class ModifiedResNet(nn.Module):
    def __init__(self):

        """
        Constructor of the ModifiedResNet class
        """

        super(ModifiedResNet, self).__init__()

        self.resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1) # in the resnet attribute we keep the original model

        self.features = nn.Sequential(*list(self.resnet.children())[:-2]) # in the features attribute we keep the part that extracts the features

        self.classifier = nn.Sequential(*list([nn.AdaptiveAvgPool2d((1, 1))] + [nn.Flatten()] +[self.resnet.fc])) # in the classifier attribute we implement the sequence of pooling and getting the final prediction

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        """
        Forward step of the model
        """

        x = self.features(x)
        x = self.classifier(x)
        return x

resnet = ModifiedResNet()
original_resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

f_ex = resnet.features
classification = resnet.classifier

resnet.eval()
original_resnet.eval();

**Quiz 1 Get the prediction from the modified model. Which class number does resnet predict?**

In [ ]:
print('Prediction of the original model: ', # Your code here)
print('Prediction of the modified model: ', # Your code here)

We get the features by applying the feature-extracting part to the input, and the classification scores by applying the classifier to the obtained features. Let us also record the sizes of the resulting map, since we will need them again.

In [ ]:
f_map = f_ex(pred) # Getting the feature map
_, N, H, W = f_map.size() # Extracting the sizes of the obtained map

c_score = classification(f_map)[0, 179] # Getting the classification score

Let us recall the formula for building GradCAM:

$$L^c_{\text{Grad-CAM}} = \mathrm{ReLU}\Big(\sum_k \alpha^c_k A^k\Big)$$

where $a^C_K$ is the average over the gradients obtained during the backward pass:

$$\alpha^c_k = \frac{1}{Z}\sum_i\sum_j \frac{\partial y^c}{\partial A^k_{ij}}$$

How do we get the gradients of the backward pass? One of the ways is to use `torch.autograd.grad()`.

In PyTorch `torch.autograd.grad()` is used to compute the gradients of some tensors with respect to other tensors — that is exactly what we need.

```
torch.autograd.grad(
    outputs,        # The tensor(s) for which the gradients have to be computed.
    inputs,         # The tensor(s) with respect to which the gradients have to be computed.)
```

**Quiz 2 Compute the gradient of `f_map` with respect to the score `c_score`. Write the result into the variable grads. How many dimensions does the output have?**

In [ ]:
grads = torch.autograd.grad(c_score, f_map)

In [ ]:
# Your code here

**Quiz 3 Implement the computation of the weights as the mean over the last two dimensions. What is the length of the result?**

Hint: you can take the mean as `grads[0].mean()`

In [ ]:
w = # Your code here

In [ ]:
gradcam = torch.matmul(w, f_map.view(N, H*W))
gradcam = gradcam.view(H, W)

gradcam = nn.functional.relu(gradcam)

In [ ]:
gradcam_tensor = gradcam.unsqueeze(0).unsqueeze(0)  # Converting it back into a tensor

interpol = F.interpolate(gradcam_tensor, (224, 224), mode="bilinear") # Interpolating the resulting map to the size we need
interpol = interpol.squeeze(0).squeeze(0) #Preparing the result for visualisation
interpol = interpol.detach().numpy()

plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.axis('off')
plt.show()

If we put everything together this way, we can write the following function

In [ ]:
def GradCAM_function(input_img, cl_sc: torch.Tensor | float, f_ex: nn.Module, classification: nn.Module) -> torch.Tensor:

    f_map = f_ex(input_img)
    _, N, H, W = f_map.size()

    c_score = classification(f_map)[0, cl_sc]

    grads = torch.autograd.grad(c_score, f_map) # we do the backward pass on the classification score with respect to the feature map
    w = grads[0][0].mean(-1).mean(-1)

    gradcam = torch.matmul(w, f_map.view(N, H*W)) # we multiply the map by the weights
    gradcam = gradcam.view(H, W)
    gradcam = nn.functional.relu(gradcam)

    return gradcam

In [ ]:
GradCAM_function(pred, 179, f_ex, classification)

And, by the definition of a counterfactual map, changing the function above a little you can build a Counterfactual GradCAM, which may or may not give you additional information.


**Quiz 4 Choose on stepik how to build a Counterfactual GradCAM based on the function for GradCAM. Using the answer, complete the function below. Did you manage to build an informative example for the target class?**

In [ ]:
def Counterfactual_GradCAM_function(input_img, cl_sc: torch.Tensor | float, f_ex: nn.Module, classification: nn.Module) -> torch.Tensor:

    f_map = f_ex(input_img)
    _, N, H, W = f_map.size()

    c_score = classification(f_map)[0, cl_sc]

    grads = torch.autograd.grad(c_score, f_map) # we do the backward pass on the classification score with respect to the feature map
    w = grads[0][0].mean(-1).mean(-1)

    gradcam = torch.matmul(w, f_map.view(N, H*W)) # COMPLETE THE CODE
    gradcam = gradcam.view(H, W)
    gradcam = nn.functional.relu(gradcam)

    return gradcam


counterfactual_gc = Counterfactual_GradCAM_function(pred, 179, f_ex, classification).unsqueeze(0).unsqueeze(0)

interpol = F.interpolate(counterfactual_gc, (224, 224), mode="bilinear") # Interpolating the resulting map to the size we need
interpol = interpol.squeeze(0).squeeze(0) #Preparing the result for visualisation
interpol = interpol.detach().numpy()

plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.axis('off')
plt.show()

**Quiz 5 In the same way, complete the class for DenseNet. Does the prediction of the modified model match the original one?**

In [ ]:
#DenseNet

class ModifiedDenseNet(nn.Module):
    def __init__(self):
        super(ModifiedDenseNet, self).__init__()
        self.densenet = models.densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)
        self.features = self.densenet.features
        self.classifier = nn.Sequential(*list([nn.AdaptiveAvgPool2d((1, 1))] + [nn.Flatten()] + [self.densenet.classifier]))

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

densenet = ModifiedDenseNet()

f_ex = densenet.features
classification = densenet.classifier

densenet.eval();

In [ ]:
print('Prediction of the original model: ', # Your code here)
print('Prediction of the modified model: ', # Your code here )

In [ ]:
gradcam = GradCAM_function(pred, 254, f_ex, classification).unsqueeze(0).unsqueeze(0)
interpol = F.interpolate(gradcam, (224, 224), mode="bilinear") # Interpolating the map onto the image
interpol = interpol.squeeze(0).squeeze(0).detach().numpy()


plt.axis('off')
plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.show()

**Way 2. Using Hooks.**

One more way to extract the outputs of the forward and backward passes is attaching hooks, which you met earlier. Let us implement the use of hooks for Densenet and ResNet.

In [ ]:
#DenseNet

# A dictionary for storing the activations and the gradients
activations = {}
gradients = {}

# The forward hook function for saving the activations
def forward_hook(module, input, output):
    activations['conv_output'] = output

# The backward hook function for saving the gradients
def backward_hook(module, grad_in, grad_out):
    gradients['conv_gradients'] = grad_out[0]


densenet = models.densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)

densenet.eval();

In [ ]:
forward_hook = densenet.features[-2].register_forward_hook(forward_hook)
backward_hook = densenet.features[-2].register_backward_hook(backward_hook)

prediction = densenet(pred)
cl_cls = prediction.argmax(dim=1)

densenet.zero_grad()

In [ ]:
prediction[0, cl_cls].backward()

In [ ]:
hook_f_map = activations['conv_output']
hook_weight = gradients['conv_gradients'].mean(axis=(2, 3)) # we get the weights as the mean over the last two dimensions


_, N, H, W = hook_f_map.size()

gradcam = torch.matmul(hook_weight, hook_f_map.view(1920, 7*7)) # we build the map by means of the product

gradcam = gradcam.view(H, W).cpu().detach().numpy() # we bring the map to a readable form


gradcam = np.maximum(gradcam, 0) #ReLU


gradcam_tensor = torch.from_numpy(gradcam).float().unsqueeze(0).unsqueeze(0)  # Converting it back into a tensor
interpol = F.interpolate(gradcam_tensor, (224, 224), mode="bilinear").squeeze(0).squeeze(0) # Interpolating the map onto the image


plt.axis('off')
plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.show()

In the same way, implement taking a Hook for ResNet, choosing the part (the **layer**) of the network you need. Did the map you got match the first map for the model?

In [ ]:
#ResNET

# A dictionary for storing the activations and the gradients
activations = {}
gradients = {}

# The forward hook function for saving the activations
def forward_hook(module, input, output):
    activations['conv_output'] = output

# The backward hook function for saving the gradients
def backward_hook(module, grad_in, grad_out):
    gradients['conv_gradients'] = grad_out[0]


resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

resnet.eval();

In [ ]:
forward_hook = resnet.layer4[-1].register_forward_hook(forward_hook)
backward_hook = resnet.layer4[-1].register_backward_hook(backward_hook)

prediction = resnet(pred) # we get the prediction
cl_cls = prediction.argmax(dim=1) # we extract the class label

resnet.zero_grad()

prediction[0, cl_cls].backward() # we do the backward pass

In [ ]:
# The same way of building the map

hook_f_map = activations['conv_output']
hook_weight = gradients['conv_gradients'].mean(axis=(2, 3))


_, N, H, W = hook_f_map.size()

gradcam = torch.matmul(hook_weight, hook_f_map.view(2048, 7*7))

gradcam = gradcam.view(H, W).cpu().detach().numpy()


gradcam = np.maximum(gradcam, 0) #ReLU


gradcam_tensor = torch.from_numpy(gradcam).float().unsqueeze(0).unsqueeze(0)  # Converting it back into a tensor
interpol = F.interpolate(gradcam_tensor, (224, 224), mode="bilinear").squeeze(0).squeeze(0) # Interpolating the map onto the image


plt.axis('off')
plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.show()

**Way 3. The library implementation.**

To finish with, let us give an example of building GradCAM from the library.

In [ ]:
def schow_library_example(model, prediction, input, original_input, target_layer):

  print('The example started...')
  print('GradCAM is buildinf for prediction: ', prediction)

  targets = [ClassifierOutputTarget(prediction)] # here we write down the class for which we are going to build the map
  target_layers = [target_layer]

  with GradCAM(model=model, target_layers=target_layers) as cam:

    grayscale_cam = cam(input_tensor=input, targets=targets)
    cam_image = show_cam_on_image(original_input, grayscale_cam[0, :], use_rgb=True) # we build the CAM on the rgb image

  cam = np.uint8(255*grayscale_cam[0, :])
  cam = cv2.merge([cam, cam, cam])
  images = np.hstack((np.uint8(255*original_input), cam , cam_image))


  return images


original_to_code_example = np.float32(display(image)) / 255

In [ ]:
resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
resnet.eval()

prediction = resnet(pred).argmax(dim=1)
images = schow_library_example(resnet, prediction, pred, original_to_code_example, resnet.layer4[-1])

Image.fromarray(images)

In [ ]:
#DenseNet
densenet = models.densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)
densenet.eval()

prediction = densenet(pred).argmax(dim=1)
images = schow_library_example(densenet, prediction, pred, original_to_code_example, densenet.features[-1])

Image.fromarray(images)

On the whole, there are many implementations of GradCAM. You can work with it
- on the basis of [other user](https://github.com/Aa-Aanegola/Grad-CAM/tree/master) implementations in pyTorch
- on the basis of [other user](https://github.com/ismailuddin/gradcam-tensorflow-2) implementations in Tensorflow
- on the basis of other open-source solutions, for example [captum](https://captum.ai/)

Our advice on how to choose — go for the one that is most convenient for you.

# **Part 4. Gradient-free: Score-CAM**

Score-CAM gives up the gradient: the weight of a channel is how much the class score grows
if you leave on the image only the region where this channel is active.

1. take the feature maps of the last convolutional layer;
2. stretch each of them to the size of the input and normalise it to $[0, 1]$ — you get a mask;
3. multiply the image by the mask and run it through the network;
4. the weight of the channel is the probability of the target class on the masked image;
5. the map is the weighted sum of the feature maps, then ReLU.

The price is visible straight from the algorithm: **one forward pass per channel**.

In [ ]:
@torch.no_grad()
def score_cam(model, x, target_layer, cls, batch=64):
    store = {}
    h = target_layer.register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    model(x)
    h.remove()

    A = store['a'][0]                                    # (C, h, w) — the feature maps
    C = A.shape[0]
    M = F.interpolate(A.unsqueeze(0), x.shape[-2:], mode='bilinear', align_corners=False)[0]
    mn = M.flatten(1).min(1).values[:, None, None]
    mx = M.flatten(1).max(1).values[:, None, None]
    M = (M - mn) / (mx - mn + 1e-8)                      # the masks in [0, 1]

    weights, passes = [], 0
    for i in range(0, C, batch):
        masked = x * M[i:i + batch].unsqueeze(1)
        # The weight of the channel is the probability of the target class on the masked image.
        weights.append(  # Your code here
        )
        passes += masked.shape[0]
    w = torch.cat(weights)

    cam = torch.relu((w[:, None, None] * A).sum(0))
    return (cam / (cam.max() + 1e-8)).cpu().numpy(), C, passes

In [ ]:
resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).eval()
cls = int(resnet(pred).argmax(1))

sc_cam, n_channels, n_passes = score_cam(resnet, pred, resnet.layer4[-1], cls)

print('channels in the last convolutional layer:', n_channels)
print('forward passes done:                     ', n_passes)

**Quiz.** How many forward passes through the network did Score-CAM perform? Give the answer in the trainer.

Compare that with Grad-CAM, for which one forward and one backward pass is enough.

# **Part 5. How to compare two maps: insertion and deletion**

**Deletion:** we remove pixels in decreasing order of importance. A good map drops the prediction quickly,
so the smaller the area under the curve, the **better**.

**Insertion:** we add the important pixels onto an empty canvas. Here **more is better**.

In [ ]:
@torch.no_grad()
def curve(model, x, cam, cls, mode='deletion', steps=50):
    H, W = x.shape[-2:]
    heat = F.interpolate(torch.tensor(cam)[None, None].float(), (H, W), mode='bilinear')[0, 0]
    order = torch.argsort(heat.flatten(), descending=True)   # in decreasing order of importance
    n, k = order.numel(), max(1, order.numel() // steps)

    ys = []
    for s in range(steps + 1):
        idx = order[:s * k]
        img = x.clone() if mode == 'deletion' else torch.zeros_like(x)
        flat = img.view(1, 3, -1)
        flat[0, :, idx] = 0 if mode == 'deletion' else x.view(1, 3, -1)[0, :, idx]
        ys.append(torch.softmax(model(flat.view_as(x)), 1)[0, cls].item())

    xs = np.linspace(0, 1, len(ys))
    # np.trapz was removed in numpy 2, np.trapezoid is not in numpy 1 — we take whatever is there
    trapz = getattr(np, 'trapezoid', None) or np.trapz
    auc =  # Your code here: the area under the curve by the trapezoidal rule
    return xs, np.array(ys), float(auc)

In [ ]:
targets = [ClassifierOutputTarget(cls)]
with GradCAM(model=resnet, target_layers=[resnet.layer4[-1]]) as cam_obj:
    gc = cam_obj(input_tensor=pred, targets=targets)[0]

for name, cmap in [('Grad-CAM', gc), ('Score-CAM', sc_cam)]:
    _, _, ad = curve(resnet, pred, cmap, cls, 'deletion')
    _, _, ai = curve(resnet, pred, cmap, cls, 'insertion')
    print(f'{name:10s}  deletion AUC {ad:.4f}   insertion AUC {ai:.4f}   '
          f'insertion − deletion {ai - ad:.4f}')

**Quiz.** Which method got the larger `insertion − deletion`? Give the answer in the trainer.

The main thing to take away: comparing maps by eye makes no sense — "it looks like the object"
and "the model uses this" are two different things. The metric answers the second question.